# Audio Fault Clustering using HDBSCAN

This notebook extracts Mel-spectrogram features from the raw `.wav` audio files, reduces their dimensionality using UMAP, and clusters them using HDBSCAN. It sweeps across various hyperparameters to find the optimal clustering that minimizes noise while discovering the distinct operational states of the machines.

In [1]:
!pip install -q hdbscan umap-learn librosa tqdm scikit-learn pandas seaborn matplotlib

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import librosa
import hdbscan
import umap.umap_ as umap
from tqdm.auto import tqdm
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import concurrent.futures

# Settings
DATA_DIR = 'archive/'
OUTPUT_CSV = 'hdbscan_cluster_results.csv'
SAMPLE_RATE = 16000
N_MELS = 128
MAX_PAD_LEN = 400
RANDOM_SEED = 42

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.
2026-04-30 14:59:46.194334: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777561186.450968      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777561186.525041      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777561187.171841      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777561187.171905      16 computation_placer.cc:177] computation placer al

## 1. Data Loading and Feature Extraction
We reuse the loading logic to extract 128x400 Mel-spectrograms. To cluster these, we compute the temporal mean and standard deviation for each Mel band, resulting in a compact 256-dimensional feature vector per audio file.

In [3]:
def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS, n_fft=1024, hop_length=512)
        log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        
        if log_mel_spec.shape[1] > MAX_PAD_LEN:
            log_mel_spec = log_mel_spec[:, :MAX_PAD_LEN]
        else:
            pad_width = MAX_PAD_LEN - log_mel_spec.shape[1]
            log_mel_spec = np.pad(log_mel_spec, pad_width=((0, 0), (0, pad_width)), mode='constant')
            
        # Extract 1D features: Mean and Std over time
        mel_mean = np.mean(log_mel_spec, axis=1)
        mel_std = np.std(log_mel_spec, axis=1)
        feature_vector = np.concatenate([mel_mean, mel_std])
        
        return file_path, feature_vector
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return file_path, None

# Discover files
wav_files = glob.glob(os.path.join(DATA_DIR, '**/*.wav'), recursive=True)
print(f"Found {len(wav_files)} audio files.")

features = []
valid_paths = []
labels = []

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(extract_features, wav_files), total=len(wav_files), desc="Extracting Features"))

for path, feat in results:
    if feat is not None:
        valid_paths.append(path)
        features.append(feat)
        
        # Extract label from path (e.g., 'archive/Machine 1/normal/... ')
        machine = [p for p in path.split(os.sep) if 'Machine' in p][0]
        state = 'Normal' if 'normal' in path.lower() else 'Abnormal'
        labels.append(f"{machine} {state}")

X = np.array(features)
print(f"Feature matrix shape: {X.shape}")

Found 0 audio files.


Extracting Features: 0it [00:00, ?it/s]

Feature matrix shape: (0,)


## 2. Dimensionality Reduction (UMAP)
HDBSCAN struggles with high dimensions (curse of dimensionality). We reduce the 256D features to 10D using UMAP.

In [4]:
print("Applying UMAP...")
reducer = umap.UMAP(n_neighbors=30, min_dist=0.0, n_components=10, random_state=RANDOM_SEED)
X_reduced = reducer.fit_transform(X)
print(f"Reduced feature matrix shape: {X_reduced.shape}")

Applying UMAP...


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

## 3. HDBSCAN Parameter Sweep & Elbow Plot
We test various `min_cluster_size` parameters to find the sweet spot that maximizes Silhouette score, minimizes noise, and discovers a meaningful number of clusters.

In [ ]:
cluster_sizes = [10, 20, 50, 100, 200, 500]
results = []

for size in tqdm(cluster_sizes, desc="Sweeping HDBSCAN Params"):
    clusterer = hdbscan.HDBSCAN(min_cluster_size=size, min_samples=size//2, gen_min_span_tree=True)
    cluster_labels = clusterer.fit_predict(X_reduced)
    
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_ratio = np.sum(cluster_labels == -1) / len(cluster_labels)
    
    if n_clusters > 1:
        # Calculate silhouette only on non-noise points
        mask = cluster_labels != -1
        if np.sum(mask) > 0 and len(set(cluster_labels[mask])) > 1:
            sil_score = silhouette_score(X_reduced[mask], cluster_labels[mask])
        else:
            sil_score = -1
    else:
        sil_score = -1
        
    results.append({
        'min_cluster_size': size,
        'n_clusters': n_clusters,
        'noise_ratio': noise_ratio,
        'silhouette_score': sil_score
    })

df_res = pd.DataFrame(results)
display(df_res)

# Plotting the 'Elbow' / Metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.lineplot(data=df_res, x='min_cluster_size', y='n_clusters', marker='o', ax=axes[0])
axes[0].set_title('Number of Clusters vs Min Cluster Size')
axes[0].set_ylabel('Number of Clusters')

sns.lineplot(data=df_res, x='min_cluster_size', y='noise_ratio', marker='o', ax=axes[1], color='red')
axes[1].set_title('Noise Ratio vs Min Cluster Size')
axes[1].set_ylabel('Noise Ratio')

sns.lineplot(data=df_res, x='min_cluster_size', y='silhouette_score', marker='o', ax=axes[2], color='green')
axes[2].set_title('Silhouette Score vs Min Cluster Size')
axes[2].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

## 4. Final Clustering & Saving Results
Based on the analysis, we pick the best `min_cluster_size` that provides solid clusters without excessive noise, and save the assignments to be used in our neural network training.

In [ ]:
# Select the best min_cluster_size
best_result = max([r for r in results if r['silhouette_score'] > 0], key=lambda x: x['silhouette_score'])
best_size = best_result['min_cluster_size']
print(f"Best min_cluster_size selected: {best_size}")

# Fit best model
best_clusterer = hdbscan.HDBSCAN(min_cluster_size=best_size, min_samples=best_size//2, gen_min_span_tree=True)
final_labels = best_clusterer.fit_predict(X_reduced)

df = pd.DataFrame({
    'file_path': valid_paths,
    'true_label': labels,
    'cluster': final_labels
})

# Make clusters pure: each cluster has only one type of class
# We keep only the majority class for each cluster, effectively filtering out "leakage"
pure_df_list = []
for c in df['cluster'].unique():
    if c == -1:
        continue # Ignore noise
    cluster_data = df[df['cluster'] == c]
    majority_class = cluster_data['true_label'].mode()[0]
    # Keep only samples matching the majority class
    pure_data = cluster_data[cluster_data['true_label'] == majority_class]
    pure_df_list.append(pure_data)

final_pure_df = pd.concat(pure_df_list, ignore_index=True)
print(f"Retained {len(final_pure_df)} purely clustered samples out of {len(df)} original samples.")

final_pure_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved pure cluster assignments to {OUTPUT_CSV}")

## 5. Visualizing Clusters with t-SNE
We'll utilize t-SNE to project our 256D features into 2D space to visualize the clusters formed and assess the purity.

In [ ]:
from sklearn.manifold import TSNE

# Generate t-SNE for plotting based on our feature matrix
print("Computing t-SNE embedding...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)

# Scatter plot
viz_df = pd.DataFrame({
    'tsne_0': X_tsne[:, 0],
    'tsne_1': X_tsne[:, 1],
    'cluster': final_labels,
    'true_label': labels
})

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Plot by True Label
sns.scatterplot(
    data=viz_df, x='tsne_0', y='tsne_1', hue='true_label',
    palette='tab10', alpha=0.7, ax=axes[0]
)
axes[0].set_title("t-SNE Colored by True Machine State")

# 2. Plot by Learned Cluster
# Plot noise (-1) smaller and gray
is_noise = viz_df['cluster'] == -1
sns.scatterplot(
    data=viz_df[is_noise], x='tsne_0', y='tsne_1',
    color='lightgray', s=10, label='Noise / -1', ax=axes[1]
)
sns.scatterplot(
    data=viz_df[~is_noise], x='tsne_0', y='tsne_1',
    hue='cluster', palette='Spectral', alpha=0.9, ax=axes[1],
    legend='full'
)
axes[1].set_title("t-SNE Colored by Found HDBSCAN Cluster")
plt.tight_layout()
plt.show()